# Etapa 6 · Primeros pasos con la placa PYNQ-Z2

**Proyecto:** IDS-IoT acelerado por hardware — *Hardware-Accelerated Passive Intrusion Detection System*
**Placa:** TUL PYNQ-Z2 (Xilinx Zynq-7000 · XC7Z020-1CLG400C)

**Objetivo de este notebook:** primer contacto con la placa. Entender el flujo de trabajo, cómo viajan
los datos desde Python hasta el hardware programable, y comprobarlo físicamente encendiendo LEDs y
evaluando compuertas lógicas con los switches y pulsadores de la placa.

> Este notebook está pensado para **ejecutarse dentro de la PYNQ-Z2** (Jupyter corre en la propia placa).
> Aun así, incluye un **simulador de escritorio**: si lo abrís en tu PC (sin `pynq` instalado) todas las
> celdas siguen corriendo y los LEDs se dibujan en texto. Así podés leer, entender y depurar el notebook
> antes de tener la placa delante.

---

## Índice

| § | Contenido |
|---|---|
| 1 | Arquitectura de la placa y flujo de trabajo |
| 2 | Cómo conectarse a la PYNQ-Z2 |
| 3 | Verificación del entorno |
| 4 | Simulador de escritorio (modo sin placa) |
| 5 | Carga del *overlay* base (bitstream) |
| 6 | Utilidades de visualización |
| 7 | Hola mundo: encender y apagar LEDs |
| 8 | Contador binario sobre los 4 LEDs |
| 9 | Lectura de switches y pulsadores |
| 10 | Compuertas lógicas: tabla de verdad y demo automática |
| 11 | Modo interactivo: los switches como entradas A y B |
| 12 | Semisumador de 1 bit (A + B → suma y acarreo) |
| 13 | Panel interactivo con `ipywidgets` |
| 14 | LEDs RGB |
| 15 | Bajo nivel: escribir registros AXI con `MMIO` |
| 16 | Buffers contiguos y DMA (puente hacia el acelerador IDS) |
| 17 | Limpieza final |
| 18 | Resumen, problemas frecuentes y próximos pasos |

## 1 · Arquitectura de la placa y flujo de trabajo

El Zynq-7020 **no** es una FPGA suelta: es un SoC con dos mitades que conviven en el mismo chip.

```
+------------------------------ XC7Z020 (Zynq-7000 SoC) ------------------------------+
|                                                                                     |
|   PS -- Processing System ------------+        +---- PL -- Programmable Logic ----+ |
|   | 2x ARM Cortex-A9 @ 650 MHz      | |        | | AXI GPIO  -> LEDs             | | |
|   | Linux (PYNQ) + Python3 + Jupyter| |  AXI   | | AXI GPIO  -> switches / btns  | | |
|   | 512 MB DDR3                     |<+------->+ | AXI DMA   -> acelerador IDS   | | |
|   | Ethernet / USB / microSD        | |        | | myproject (hls4ml) <- Etapa 5 | | |
|   +---------------------------------+ |        | +-------------------------------+ | |
|            ^                          |        |            ^                      | |
+------------|--------------------------+--------+------------|----------------------+ |
             |  aca escribis Python                            |  esto lo define el .bit
             +-------------------------------------------------+
```

* **PS (Processing System):** el ARM corriendo Linux. Ahí vive Jupyter, ahí se ejecuta este notebook,
  ahí está NumPy. Es *software*.
* **PL (Programmable Logic):** la tela FPGA. Está **vacía** hasta que se le carga un *bitstream*.
  Es *hardware* configurado en tiempo de ejecución.

### El flujo de trabajo completo

```
  PC de desarrollo                          PYNQ-Z2
 +------------------+                 +--------------------------------------+
 | Keras / QKeras   |                 |                                      |
 |       |          |                 |  Overlay("diseño.bit")               |
 | hls4ml (Etapa 5) |   .bit + .hwh   |       |  carga el bitstream en la PL  |
 |       |          |  ------------>  |  base.ip_dict -> drivers Python      |
 | Vitis HLS -> IP  |   scp / samba   |       |                              |
 |       |          |                 |  datos <-> hardware (MMIO / DMA)     |
 | Vivado -> .bit   |                 |       |                              |
 +------------------+                 |  resultados en NumPy / LEDs          |
                                      +--------------------------------------+
```

1. En el PC se entrena y comprime el modelo (Etapas 2–4).
2. `hls4ml` + Vitis HLS lo convierten en un IP core; Vivado lo integra y genera **`diseño.bit`**
   (la configuración del hardware) y **`diseño.hwh`** (la descripción de qué IPs hay y en qué
   direcciones). **Los dos archivos deben tener el mismo nombre base y estar en la misma carpeta.**
3. Se copian a la placa y `pynq.Overlay(...)` los carga: PYNQ lee el `.hwh`, arma un diccionario de IPs
   (`ip_dict`) y expone drivers Python listos para usar.
4. Desde Python se mueven datos hacia y desde el hardware.

### Los tres caminos de datos hacia la PL

| Camino | Cómo se usa | Para qué sirve | Dónde se ve acá |
|---|---|---|---|
| **AXI-Lite / MMIO** | `MMIO(dir).write(offset, valor)` | Registros de control y datos chicos: LEDs, switches, `start`/`done` del acelerador | §7 a §15 |
| **AXI-Stream + DMA** | `pynq.allocate()` + `dma.sendchannel.transfer()` | Bloques grandes: los lotes de flujos de red del IDS | §16 |
| **Interrupciones / GPIO** | `pin.wait_for_value()` | Eventos: "terminé", "se pulsó un botón" | §11 |

Los LEDs son la excusa perfecta: usan exactamente el **mismo mecanismo AXI** que después va a usar el
acelerador de inferencia, pero con un resultado que se ve con los ojos.

### Recursos físicos de la PYNQ-Z2

| Recurso | Cantidad | Nombre en PYNQ | Notas |
|---|---|---|---|
| LEDs verdes | 4 | `base.leds` → `LD0..LD3` | salida, 4 bits |
| LEDs RGB | 2 | `base.rgbleds[4]`, `base.rgbleds[5]` | `LD4`, `LD5`, 3 bits de color |
| Switches deslizantes | 2 | `base.switches` → `SW0`, `SW1` | entrada, 2 bits |
| Pulsadores | 4 | `base.buttons` → `BTN0..BTN3` | entrada, 4 bits, momentáneos |

## 2 · Cómo conectarse a la PYNQ-Z2

Checklist de arranque (una sola vez):

1. **Jumper `JP4` (boot)** → posición **`SD`** (no `JTAG`, no `QSPI`).
2. **Jumper `JP5` (alimentación)** → `USB` si alimentás por micro-USB, o `REG` si usás la fuente de 12 V.
   Para trabajar con periféricos y bitstreams grandes, la fuente externa es más segura.
3. **Micro-SD** con la imagen oficial PYNQ para PYNQ-Z2 (`pynq_z2_vX.Y.img`) insertada.
4. Cable **micro-USB** al PC (alimentación + puerto serie) y cable **Ethernet**.
5. Encender con el interruptor. El LED rojo de *power* enciende de inmediato; el LED verde `DONE`
   enciende cuando se cargó el bitstream de arranque. Después parpadean los 4 LEDs verdes y las dos
   RGB: eso significa **Linux arrancado y PYNQ listo**.

### Acceso por red

| Escenario | Dirección | Qué configurar en el PC |
|---|---|---|
| Cable Ethernet directo PC ↔ placa | `http://192.168.2.99:9090` | IP estática `192.168.2.1`, máscara `255.255.255.0` |
| Placa conectada al router (DHCP) | `http://pynq:9090` o la IP que asigne el router | nada |

* **Jupyter:** `http://192.168.2.99:9090` — usuario/contraseña por defecto: **`xilinx` / `xilinx`**.
* **SSH:** `ssh xilinx@192.168.2.99` (misma contraseña).
* **Samba:** `\\192.168.2.99\xilinx` en el explorador de Windows — la forma más cómoda de copiar este
  notebook, los `.bit` y los `.hwh` a la placa.

### Subir este notebook

Arrastrarlo por Samba a `\\192.168.2.99\xilinx\jupyter_notebooks\`, o por consola desde el PC:

```
scp 08a_pynq_z2_primeros_pasos.ipynb xilinx@192.168.2.99:/home/xilinx/jupyter_notebooks/
```

Después se abre desde el Jupyter de la placa y se ejecuta celda por celda con `Shift+Enter`.

> **Seguridad:** la imagen PYNQ trae credenciales por defecto y el servicio Samba abierto. Está bien
> para banco de trabajo aislado; no la expongas a la red del laboratorio sin cambiar la contraseña
> (`passwd` por SSH).

## 3 · Verificación del entorno

Primero averiguamos **dónde se está ejecutando este notebook**. Si `platform.machine()` devuelve
`armv7l`, estamos dentro del ARM de la placa. Si devuelve `AMD64` o `x86_64`, estamos en el PC y el
notebook va a usar el simulador de la §4.

In [ ]:
# =====================================================================
# 3. VERIFICACION DEL ENTORNO DE EJECUCION
# =====================================================================
import os
import sys
import time
import platform

try:
    import pynq
    from pynq import Overlay, MMIO, allocate
    from pynq.overlays.base import BaseOverlay
    EN_PLACA = True
    _motivo_sin_pynq = None
except Exception as e:          # ImportError en el PC; otros errores si falta el driver
    pynq = None
    EN_PLACA = False
    _motivo_sin_pynq = repr(e)


def _leer_modelo_placa():
    ruta = "/proc/device-tree/model"
    if os.path.exists(ruta):
        with open(ruta, "rb") as f:
            return f.read().decode(errors="ignore").strip("\x00").strip()
    return "no disponible (el equipo no tiene device-tree)"


print("=" * 64)
print(" ENTORNO DE EJECUCION")
print("=" * 64)
print(f"  Python        : {platform.python_version()}")
print(f"  Arquitectura  : {platform.machine()}")
print(f"  Sistema       : {platform.system()} {platform.release()}")
print(f"  Hostname      : {platform.node()}")
print(f"  Modelo (DT)   : {_leer_modelo_placa()}")
print(f"  PYNQ          : {pynq.__version__ if EN_PLACA else 'NO INSTALADO'}")
print("-" * 64)
if EN_PLACA:
    print("  MODO: PLACA REAL  ->  las celdas escriben hardware fisico.")
else:
    print("  MODO: SIMULACION  ->  no hay 'pynq' en este interprete.")
    print(f"        motivo: {_motivo_sin_pynq}")
    print("        Los LEDs se dibujan en texto; el resto del codigo es identico.")
print("=" * 64)

## 4 · Simulador de escritorio (modo sin placa)

Estas clases **imitan la API real de PYNQ** (`AxiGPIO.Channel`, `RGBLED`) con el subconjunto que usa
este notebook: `read()`, `write(valor, mascara)`, indexación por bit con `on()` / `off()` / `toggle()`.

Si estás ejecutando en la placa, la celda define las clases pero **no se usan**. Es la única parte del
notebook que no existe en un flujo PYNQ normal: está sólo para poder estudiar el código sin hardware.

In [ ]:
# =====================================================================
# 4. SIMULADOR DE ESCRITORIO (solo se usa si EN_PLACA == False)
# =====================================================================
class _SimBit:
    """Emula el objeto que devuelve AxiGPIO.Channel[i] (un unico bit)."""

    def __init__(self, banco, idx):
        self._banco = banco
        self._idx = idx

    def on(self):
        self._banco._valor |= (1 << self._idx)

    def off(self):
        self._banco._valor &= ~(1 << self._idx)

    def toggle(self):
        self._banco._valor ^= (1 << self._idx)

    def read(self):
        return (self._banco._valor >> self._idx) & 1

    def write(self, valor):
        self.on() if valor else self.off()


class _SimGPIO:
    """Emula pynq.lib.AxiGPIO.Channel: un banco de N bits."""

    def __init__(self, ancho, valor=0):
        self._ancho = ancho
        self._valor = valor

    def __len__(self):
        return self._ancho

    def __getitem__(self, idx):
        if not 0 <= idx < self._ancho:
            raise IndexError(f"bit {idx} fuera de rango (0..{self._ancho - 1})")
        return _SimBit(self, idx)

    def read(self):
        return self._valor & ((1 << self._ancho) - 1)

    def write(self, valor, mascara):
        self._valor = (self._valor & ~mascara) | (valor & mascara)

    def simular_entrada(self, valor):
        """Exclusivo del simulador: fuerza el valor de una entrada."""
        self._valor = valor & ((1 << self._ancho) - 1)


class _SimRGB:
    """Emula pynq.lib.RGBLED."""

    def __init__(self, indice):
        self.indice = indice
        self._color = 0

    def on(self, color):
        self._color = color & 0b111

    def off(self):
        self._color = 0

    def read(self):
        return self._color


class _SimBaseOverlay:
    """Emula lo justo de pynq.overlays.base.BaseOverlay para la PYNQ-Z2."""

    def __init__(self):
        self.leds = _SimGPIO(4)
        self.switches = _SimGPIO(2)
        self.buttons = _SimGPIO(4)
        self.rgbleds = [None] * 4 + [_SimRGB(4), _SimRGB(5)]
        # ip_dict con la misma forma (parcial) que el real
        self.ip_dict = {
            "rgbleds_gpio":  {"phys_addr": 0x41240000, "addr_range": 0x10000,
                              "type": "xilinx.com:ip:axi_gpio:2.0"},
            "leds_gpio":     {"phys_addr": 0x41200000, "addr_range": 0x10000,
                              "type": "xilinx.com:ip:axi_gpio:2.0"},
            "switches_gpio": {"phys_addr": 0x41210000, "addr_range": 0x10000,
                              "type": "xilinx.com:ip:axi_gpio:2.0"},
            "btns_gpio":     {"phys_addr": 0x41220000, "addr_range": 0x10000,
                              "type": "xilinx.com:ip:axi_gpio:2.0"},
        }

    def is_loaded(self):
        return True


print("Clases del simulador definidas.")
print("Se instancian en la seccion 5 unicamente si EN_PLACA == False.")

## 5 · Carga del *overlay* base

`BaseOverlay("base.bit")` hace tres cosas, en este orden:

1. **Escribe el bitstream en la PL** a través del puerto `PCAP` del PS (tarda unas décimas de segundo).
   A partir de ese instante la tela FPGA *es* el circuito descrito por `base.bit`.
2. **Parsea `base.hwh`** para saber qué IPs existen, en qué dirección física está cada una y qué
   registros tiene. Eso es lo que llena `ip_dict`.
3. **Instancia drivers Python** y los cuelga como atributos: `base.leds`, `base.switches`,
   `base.buttons`, `base.rgbleds`, además de los IOP de los Pmod/Arduino, audio, HDMI, etc.

El *overlay base* es el diseño de fábrica: trae los AXI GPIO cableados a los LEDs, switches y
pulsadores. Más adelante vas a reemplazar `base.bit` por tu propio bitstream con el IP de `hls4ml`,
y el procedimiento va a ser **exactamente el mismo**.

In [ ]:
# =====================================================================
# 5. CARGA DEL OVERLAY (BITSTREAM -> LOGICA PROGRAMABLE)
# =====================================================================
t0 = time.time()

if EN_PLACA:
    base = BaseOverlay("base.bit")          # descarga el bitstream a la PL
else:
    base = _SimBaseOverlay()                # sustituto para el PC

t_carga = time.time() - t0

# Atajos que se usan en todo el notebook
leds     = base.leds        # 4 bits de salida  -> LD0..LD3
switches = base.switches    # 2 bits de entrada -> SW0, SW1
buttons  = base.buttons     # 4 bits de entrada -> BTN0..BTN3
rgbleds  = base.rgbleds     # lista: los indices validos son 4 y 5

print(f"Overlay cargado en {t_carga:.3f} s")
print(f"  LEDs     : {len(leds)} bits")
print(f"  Switches : {len(switches)} bits")
print(f"  Botones  : {len(buttons)} bits")
print(f"  RGB LEDs : indices validos {[i for i, r in enumerate(rgbleds) if r is not None]}")

print("\nIPs presentes en la logica programable (ip_dict):")
print(f"{'nombre':<26} {'direccion':>12} {'rango':>10}   tipo")
print("-" * 90)
for nombre, info in sorted(base.ip_dict.items()):
    direccion = info.get("phys_addr")
    rango = info.get("addr_range")
    tipo = info.get("type", "")
    txt_dir = f"0x{direccion:08X}" if isinstance(direccion, int) else str(direccion)
    txt_rng = f"0x{rango:X}" if isinstance(rango, int) else str(rango)
    print(f"{nombre:<26} {txt_dir:>12} {txt_rng:>10}   {tipo}")

> **Leé la tabla de arriba con atención.** Cada fila es un bloque de hardware *real* que existe ahora
> mismo dentro del chip, y `phys_addr` es la dirección de memoria por la que el ARM le habla.
> Cuando cargues el overlay del IDS, en esta misma tabla van a aparecer tu `axi_dma_0` y tu
> `ids_axi_0`. Todo lo que sigue es aprender a hablarle a esas direcciones.

## 6 · Utilidades de visualización

Dos ayudas para el resto del notebook:

* `escribir_leds(valor)` — escribe los 4 bits de golpe (`leds.write(valor, 0xF)`) y guarda el estado.
* `panel(...)` — dibuja en texto el estado de LEDs, switches y botones, para que en el *output* del
  notebook quede registro de lo que estaba pasando en la placa.

Ojo con el orden de los bits: **`LD0` es el bit menos significativo**, así que en el dibujo los LEDs
van de `LD3` (izquierda) a `LD0` (derecha), igual que se escribe un número binario.

In [ ]:
# =====================================================================
# 6. UTILIDADES DE VISUALIZACION
# =====================================================================
ENC, APA = "●", "○"      # circulo lleno / circulo vacio
ESTADO_LEDS = 0


def escribir_leds(valor):
    """Escribe los 4 LEDs de una sola vez. Devuelve el valor efectivamente escrito."""
    global ESTADO_LEDS
    valor &= 0xF
    leds.write(valor, 0xF)         # (valor, mascara) -> API real de AxiGPIO
    ESTADO_LEDS = valor
    return valor


def dibujar_bits(valor, n, prefijo):
    """Devuelve dos lineas de texto (etiquetas y estado), del bit n-1 al bit 0."""
    etiquetas = " ".join(f"{prefijo + str(i):^5}" for i in reversed(range(n)))
    simbolos = " ".join(f"{(ENC if (valor >> i) & 1 else APA):^5}" for i in reversed(range(n)))
    return etiquetas, simbolos


def panel(titulo="", valor_leds=None, mostrar_entradas=True):
    """Dibuja en texto el estado completo de la placa."""
    v = ESTADO_LEDS if valor_leds is None else (valor_leds & 0xF)
    if titulo:
        print(titulo)
    e, s = dibujar_bits(v, 4, "LD")
    print(f"  LEDs      {e}    0b{v:04b} = {v:2d}")
    print(f"            {s}")
    if mostrar_entradas:
        sw, bt = switches.read(), buttons.read()
        e, s = dibujar_bits(sw, 2, "SW")
        print(f"  Switches  {e}    0b{sw:02b}")
        print(f"            {s}")
        e, s = dibujar_bits(bt, 4, "BTN")
        print(f"  Botones   {e}    0b{bt:04b}")
        print(f"            {s}")


escribir_leds(0b0000)
panel("Estado inicial de la placa:")

## 7 · Hola mundo: encender y apagar LEDs

Dos formas de escribir los LEDs, y conviene entender la diferencia porque se repite con cualquier IP:

| Forma | Código | Qué hace por debajo |
|---|---|---|
| **Bit a bit** | `leds[0].on()` | lee-modifica-escribe el registro `GPIO_DATA` (una transacción AXI por operación) |
| **Banco completo** | `leds.write(0b1010, 0xF)` | una sola escritura del registro, con máscara |

Para 4 LEDs da igual; para un acelerador que recibe decenas de *features* por paquete, escribir de a un
registro por vez es exactamente lo que hay que evitar. Ese es el motivo por el que el IDS usa DMA (§16).

**Mirá la placa mientras corre esta celda:** los cuatro LEDs verdes se encienden en secuencia.

In [ ]:
# =====================================================================
# 7. HOLA MUNDO: LEDS UNO POR UNO
# =====================================================================
escribir_leds(0)

print("Barrido de LD0 a LD3 (forma bit a bit)\n")
for i in range(len(leds)):
    leds[i].on()                       # <-- API individual de PYNQ
    ESTADO_LEDS |= (1 << i)            # (solo para que el dibujo acompane)
    panel(f"  paso {i}: leds[{i}].on()", mostrar_entradas=False)
    time.sleep(0.35)

time.sleep(0.4)

print("\nApagado en orden inverso\n")
for i in reversed(range(len(leds))):
    leds[i].off()
    ESTADO_LEDS &= ~(1 << i)
    panel(f"  paso {i}: leds[{i}].off()", mostrar_entradas=False)
    time.sleep(0.35)

print("\nAhora la forma de banco completo: leds.write(valor, mascara)")
for patron in (0b1010, 0b0101, 0b1111, 0b0000):
    escribir_leds(patron)
    panel(f"  leds.write(0b{patron:04b}, 0xF)", mostrar_entradas=False)
    time.sleep(0.5)

## 8 · Contador binario sobre los 4 LEDs

El clásico: contar de 0 a 15 en binario. Sirve para verificar que los cuatro bits responden y que el
orden `LD3 LD2 LD1 LD0` es el que esperás.

Cambiá `PAUSA` si lo querés más lento (`0.5`) o como un destello (`0.05`).

In [ ]:
# =====================================================================
# 8. CONTADOR BINARIO 0..15
# =====================================================================
from IPython.display import clear_output

PAUSA = 0.25      # segundos entre incrementos
VUELTAS = 1       # cuantas veces recorrer 0..15

for vuelta in range(VUELTAS):
    for n in range(16):
        escribir_leds(n)
        clear_output(wait=True)
        print(f"CONTADOR BINARIO   (vuelta {vuelta + 1}/{VUELTAS})\n")
        panel(valor_leds=n, mostrar_entradas=False)
        time.sleep(PAUSA)

escribir_leds(0)
clear_output(wait=True)
print("Contador finalizado. LEDs apagados.\n")
panel(mostrar_entradas=False)

## 9 · Lectura de switches y pulsadores

Hasta ahora sólo escribimos. Ahora leemos: switches y pulsadores son AXI GPIO configurados como
**entrada**, y `read()` devuelve el valor del pin en ese instante.

* `switches.read()` → entero de 2 bits (`SW1 SW0`).
* `buttons.read()` → entero de 4 bits (`BTN3..BTN0`); **1 = pulsado**, vuelve a 0 al soltar.
* `buttons[0].wait_for_value(1)` → bloquea la ejecución hasta que se pulse `BTN0`. Esto ya es manejo
  de eventos, y es el mismo patrón que se usa para esperar el `done` de un acelerador.

La celda siguiente hace de **espejo**: durante unos segundos copia el estado de los switches a `LD0`/`LD1`
y el de los botones a `LD2`/`LD3`. Movelos mientras corre.

In [ ]:
# =====================================================================
# 9. LECTURA DE ENTRADAS: SWITCHES Y PULSADORES
# =====================================================================
print("Lectura instantanea:")
print(f"  switches.read()    = 0b{switches.read():02b}")
print(f"  buttons.read()     = 0b{buttons.read():04b}")
print(f"  switches[0].read() = {switches[0].read()}   (SW0 aislado)")

# --- Espejo entradas -> LEDs -----------------------------------------
DURACION = 12.0      # segundos que dura el espejo
PERIODO = 0.08       # cada cuanto se muestrea (~12 Hz)

print(f"\nEspejo activo {DURACION:.0f} s: move SW0/SW1 y apreta BTN0/BTN1.")
print("  LD0 <- SW0 | LD1 <- SW1 | LD2 <- BTN0 | LD3 <- BTN1\n")

t_ini = time.time()
while time.time() - t_ini < DURACION:
    if EN_PLACA:
        sw = switches.read()
        bt = buttons.read()
    else:
        # En simulacion generamos una secuencia para ver el mecanismo
        paso = int(time.time() - t_ini) % 4
        sw = paso
        bt = ((paso & 1) << 1) | (paso >> 1)
        switches.simular_entrada(sw)
        buttons.simular_entrada(bt)

    valor = (sw & 0b11) | ((bt & 0b11) << 2)
    escribir_leds(valor)

    clear_output(wait=True)
    restante = DURACION - (time.time() - t_ini)
    print(f"ESPEJO ENTRADAS -> LEDS      (quedan {restante:4.1f} s)\n")
    panel(valor_leds=valor)
    time.sleep(PERIODO)

escribir_leds(0)
clear_output(wait=True)
print("Espejo finalizado.\n")
panel()

## 10 · Compuertas lógicas: tabla de verdad y demo automática

Acá está el corazón del pedido: **combinaciones lógicas reflejadas en los LEDs**.

El esquema de asignación que vamos a usar en esta sección y en la siguiente:

```
  LD3        LD2        LD1        LD0
 SALIDA    (apagado)   B = SW1    A = SW0
```

Las compuertas se evalúan **en software, en el ARM**. Es importante ser honesto con esto: en este
notebook la lógica corre en Python y el hardware sólo muestra el resultado. Implementar la compuerta
*dentro de la FPGA* (que `LD3` se encienda por cableado, sin que el procesador intervenga) requiere un
overlay propio hecho en Vivado — es el paso natural siguiente, y lo dejamos anotado en §18.

La celda siguiente define el diccionario de compuertas e imprime la tabla de verdad completa.

In [ ]:
# =====================================================================
# 10. COMPUERTAS LOGICAS: DEFINICION Y TABLA DE VERDAD
# =====================================================================
COMPUERTAS = {
    "AND":   lambda a, b: a & b,
    "OR":    lambda a, b: a | b,
    "XOR":   lambda a, b: a ^ b,
    "NAND":  lambda a, b: 1 - (a & b),
    "NOR":   lambda a, b: 1 - (a | b),
    "XNOR":  lambda a, b: 1 - (a ^ b),
    "NOT A": lambda a, b: 1 - a,
    "IMPL":  lambda a, b: (1 - a) | b,      # A -> B  (implicacion logica)
}

# Un color de LED RGB por compuerta, para identificarla de un vistazo (ver seccion 14)
COLOR_COMPUERTA = {
    "AND":   0b100,   # rojo
    "OR":    0b010,   # verde
    "XOR":   0b001,   # azul
    "NAND":  0b110,   # amarillo
    "NOR":   0b011,   # cian
    "XNOR":  0b101,   # magenta
    "NOT A": 0b111,   # blanco
    "IMPL":  0b010,   # verde
}


def tabla_de_verdad(nombres=None):
    """Imprime la tabla de verdad de las compuertas indicadas."""
    nombres = list(COMPUERTAS) if nombres is None else nombres
    cab = "  A  B |" + "".join(f"{n:^8}" for n in nombres)
    print(cab)
    print("  " + "-" * (len(cab) - 2))
    for a in (0, 1):
        for b in (0, 1):
            fila = f"  {a}  {b} |"
            fila += "".join(f"{COMPUERTAS[n](a, b):^8}" for n in nombres)
            print(fila)


print("TABLA DE VERDAD (A = SW0, B = SW1)\n")
tabla_de_verdad()
print("\nCompuertas disponibles:", ", ".join(COMPUERTAS))

### 10.b · Demo automática

Esta celda recorre sola las cuatro combinaciones de entrada (`00`, `01`, `10`, `11`) y las muestra en
los LEDs. **No hace falta tocar la placa**: sirve para verificar el cableado y para ver la compuerta
"en movimiento". Cambiá `COMPUERTA` por cualquier clave del diccionario.

In [ ]:
# =====================================================================
# 10.b DEMO AUTOMATICA DE UNA COMPUERTA
# =====================================================================
COMPUERTA = "XOR"       # probá con: AND, OR, XOR, NAND, NOR, XNOR, NOT A, IMPL
PAUSA = 1.2             # segundos por combinacion


def mapear_a_leds(a, b, salida):
    """LD0 = A, LD1 = B, LD2 = 0, LD3 = salida."""
    return (a & 1) | ((b & 1) << 1) | ((salida & 1) << 3)


def demo_compuerta(nombre, pausa=PAUSA, mostrar_rgb=True):
    f = COMPUERTAS[nombre]
    for a in (0, 1):
        for b in (0, 1):
            s = f(a, b)
            valor = mapear_a_leds(a, b, s)
            escribir_leds(valor)
            if mostrar_rgb and rgbleds[4] is not None:
                rgbleds[4].on(COLOR_COMPUERTA.get(nombre, 0b111))

            clear_output(wait=True)
            print(f"COMPUERTA {nombre}\n")
            print(f"   A = {a}   B = {b}   ->   {nombre}(A,B) = {s}")
            print(f"   {'ENCENDIDO' if s else 'apagado'}: LD3\n")
            panel(mostrar_entradas=False)
            print("\n   LD0 = A    LD1 = B    LD2 = -    LD3 = SALIDA")
            time.sleep(pausa)


demo_compuerta(COMPUERTA)
print("\nDemo terminada. Cambia COMPUERTA arriba y volve a ejecutar la celda.")

## 11 · Modo interactivo: los switches como entradas A y B

Ahora sí, **vos sos las entradas**. El bucle muestrea la placa unas 20 veces por segundo:

| Control | Función |
|---|---|
| `SW0` | entrada **A** |
| `SW1` | entrada **B** |
| `BTN0` | pasa a la **siguiente compuerta** (detección de flanco, un cambio por pulsación) |
| `BTN1` | vuelve a la compuerta **anterior** |
| `BTN3` | **sale** del bucle |

Salida en `LD3`, entradas espejadas en `LD0`/`LD1`, y el LED RGB `LD4` indica con su color qué
compuerta está activa.

> Si el bucle queda colgado, `Kernel → Interrupt` (o el botón ■) lo corta; el bucle también tiene un
> **timeout** de `SEGUNDOS`.

In [ ]:
# =====================================================================
# 11. MODO INTERACTIVO: COMPUERTAS CON SWITCHES Y BOTONES
# =====================================================================
SEGUNDOS = 60.0      # timeout de seguridad
PERIODO = 0.05       # 20 Hz de muestreo


def bucle_compuertas(segundos=SEGUNDOS, periodo=PERIODO):
    nombres = list(COMPUERTAS)
    idx = 0
    btn_prev = 0
    t_ini = time.time()
    motivo = "timeout"

    while time.time() - t_ini < segundos:
        bt = buttons.read()
        flancos = bt & ~btn_prev          # bits que pasaron de 0 a 1 (antirrebote simple)
        btn_prev = bt

        if flancos & 0b1000:              # BTN3 -> salir
            motivo = "BTN3 pulsado"
            break
        if flancos & 0b0001:              # BTN0 -> siguiente compuerta
            idx = (idx + 1) % len(nombres)
        if flancos & 0b0010:              # BTN1 -> compuerta anterior
            idx = (idx - 1) % len(nombres)

        nombre = nombres[idx]
        sw = switches.read()
        a, b = sw & 1, (sw >> 1) & 1
        s = COMPUERTAS[nombre](a, b)

        escribir_leds(mapear_a_leds(a, b, s))
        if rgbleds[4] is not None:
            rgbleds[4].on(COLOR_COMPUERTA.get(nombre, 0b111))

        clear_output(wait=True)
        restante = segundos - (time.time() - t_ini)
        print(f"MODO INTERACTIVO   compuerta [{idx + 1}/{len(nombres)}]: {nombre}"
              f"        (timeout en {restante:4.1f} s)\n")
        print(f"   A = {a}   B = {b}   ->   {nombre} = {s}\n")
        panel(mostrar_entradas=False)
        print("\n   LD0 = A   LD1 = B   LD3 = SALIDA")
        print("   BTN0 siguiente | BTN1 anterior | BTN3 salir")
        time.sleep(periodo)

    return motivo


if EN_PLACA:
    motivo = bucle_compuertas()
    print(f"\nBucle finalizado ({motivo}).")
else:
    print("SIMULACION: no hay switches fisicos, se recorren todas las compuertas automaticamente.\n")
    time.sleep(0.5)
    for n in COMPUERTAS:
        demo_compuerta(n, pausa=0.35)
    print("\nRecorrido de todas las compuertas terminado.")

escribir_leds(0)
if rgbleds[4] is not None:
    rgbleds[4].off()

## 12 · Semisumador de 1 bit (A + B)

Combinar dos compuertas ya es un circuito útil. El *half adder* es el bloque básico de cualquier
aritmética digital:

```
   SUMA    = A XOR B      -> LD2
   ACARREO = A AND  B     -> LD3
```

Nuevo mapeo de LEDs para esta sección:

```
  LD3        LD2        LD1        LD0
 ACARREO    SUMA       B = SW1    A = SW0
```

Con `A=1` y `B=1` vas a ver `LD2` apagado y `LD3` encendido: `1 + 1 = 10` en binario. Ese "2" leído en
`LD3 LD2` es exactamente lo que hace un sumador dentro de la FPGA.

In [ ]:
# =====================================================================
# 12. SEMISUMADOR DE 1 BIT
# =====================================================================
def semisumador(a, b):
    """Devuelve (suma, acarreo) para dos bits."""
    return (a ^ b), (a & b)


def demo_semisumador(pausa=1.2):
    for a in (0, 1):
        for b in (0, 1):
            suma, acarreo = semisumador(a, b)
            valor = (a & 1) | ((b & 1) << 1) | (suma << 2) | (acarreo << 3)
            escribir_leds(valor)

            clear_output(wait=True)
            print("SEMISUMADOR DE 1 BIT   (A + B)\n")
            print(f"   {a} + {b} = 0b{acarreo}{suma}  =  {acarreo * 2 + suma}")
            print(f"   suma (A XOR B) = {suma}      acarreo (A AND B) = {acarreo}\n")
            panel(mostrar_entradas=False)
            print("\n   LD0 = A   LD1 = B   LD2 = SUMA   LD3 = ACARREO")
            time.sleep(pausa)


demo_semisumador()

print("\nTabla completa:")
print("   A  B | SUMA  ACARREO")
print("   -----+---------------")
for a in (0, 1):
    for b in (0, 1):
        s, c = semisumador(a, b)
        print(f"   {a}  {b} |  {s}      {c}")

### 12.b · Semisumador con los switches (opcional)

Igual que §11 pero con el semisumador. Movete `SW0` y `SW1`; `BTN3` corta el bucle.

In [ ]:
# =====================================================================
# 12.b SEMISUMADOR INTERACTIVO
# =====================================================================
def bucle_semisumador(segundos=45.0, periodo=0.05):
    t_ini = time.time()
    btn_prev = 0
    while time.time() - t_ini < segundos:
        bt = buttons.read()
        if (bt & ~btn_prev) & 0b1000:
            print("BTN3 pulsado: salgo.")
            break
        btn_prev = bt

        sw = switches.read()
        a, b = sw & 1, (sw >> 1) & 1
        suma, acarreo = semisumador(a, b)
        escribir_leds((a) | (b << 1) | (suma << 2) | (acarreo << 3))

        clear_output(wait=True)
        print(f"SEMISUMADOR INTERACTIVO      (timeout en {segundos - (time.time() - t_ini):4.1f} s)\n")
        print(f"   {a} + {b} = 0b{acarreo}{suma} = {acarreo * 2 + suma}\n")
        panel(mostrar_entradas=False)
        print("\n   LD0 = A   LD1 = B   LD2 = SUMA   LD3 = ACARREO   |   BTN3 = salir")
        time.sleep(periodo)


if EN_PLACA:
    bucle_semisumador()
    escribir_leds(0)
else:
    print("SIMULACION: se omite el bucle interactivo (no hay switches fisicos).")

## 13 · Panel interactivo con `ipywidgets`

Si no tenés la placa a mano (o querés forzar combinaciones sin estirarte hasta los switches), los
mismos LEDs se pueden manejar desde el navegador. Los widgets corren en el cliente, mandan el evento al
kernel de la placa y el kernel escribe el AXI GPIO: **es el mismo camino de datos**, sólo cambia quién
origina el estímulo.

In [ ]:
# =====================================================================
# 13. PANEL INTERACTIVO (ipywidgets)
# =====================================================================
try:
    import ipywidgets as widgets
    from IPython.display import display

    w_a = widgets.ToggleButton(value=False, description="A = 0", button_style="")
    w_b = widgets.ToggleButton(value=False, description="B = 0", button_style="")
    w_gate = widgets.Dropdown(options=list(COMPUERTAS), value="XOR", description="Compuerta:")
    salida = widgets.Output()

    def actualizar(_=None):
        a = int(w_a.value)
        b = int(w_b.value)
        w_a.description = f"A = {a}"
        w_b.description = f"B = {b}"
        w_a.button_style = "success" if a else ""
        w_b.button_style = "success" if b else ""

        nombre = w_gate.value
        s = COMPUERTAS[nombre](a, b)
        escribir_leds(mapear_a_leds(a, b, s))
        if rgbleds[4] is not None:
            rgbleds[4].on(COLOR_COMPUERTA.get(nombre, 0b111))

        with salida:
            salida.clear_output(wait=True)
            print(f"{nombre}({a}, {b}) = {s}\n")
            panel(mostrar_entradas=False)
            print("\n   LD0 = A   LD1 = B   LD3 = SALIDA")

    for w in (w_a, w_b, w_gate):
        w.observe(actualizar, names="value")

    display(widgets.HBox([w_a, w_b, w_gate]), salida)
    actualizar()

except ImportError:
    print("ipywidgets no esta instalado en este entorno; se omite el panel interactivo.")
    print("En la imagen oficial de PYNQ viene incluido.")

## 14 · LEDs RGB

`LD4` y `LD5` son LEDs RGB: tres bits por LED, uno por canal. La API de PYNQ los expone como
`base.rgbleds[4]` y `base.rgbleds[5]` (los índices 0 a 3 valen `None` porque corresponden a los LEDs
verdes simples).

Convención de bits en `pynq.lib.RGBLED`: **bit0 = azul, bit1 = verde, bit2 = rojo**.

> Si en tu placa el rojo y el azul aparecen intercambiados, invertí el mapeo del diccionario `COLORES`:
> depende de la revisión del hardware y del overlay, y es una comprobación empírica de 10 segundos.

In [ ]:
# =====================================================================
# 14. LEDS RGB (LD4 y LD5)
# =====================================================================
COLORES = {
    "apagado":  0b000,
    "azul":     0b001,
    "verde":    0b010,
    "cian":     0b011,
    "rojo":     0b100,
    "magenta":  0b101,
    "amarillo": 0b110,
    "blanco":   0b111,
}

if rgbleds[4] is None or rgbleds[5] is None:
    print("Este overlay no expone LEDs RGB.")
else:
    print("Recorriendo la paleta en LD4 y LD5 (mira la placa)...\n")
    for nombre, codigo in COLORES.items():
        rgbleds[4].on(codigo)
        rgbleds[5].on(codigo ^ 0b111)      # el complementario en LD5
        print(f"  LD4 = {nombre:<9} (0b{codigo:03b})    LD5 = complementario (0b{codigo ^ 0b111:03b})")
        time.sleep(0.6)

    rgbleds[4].off()
    rgbleds[5].off()
    print("\nRGB apagados.")

# --- Uso practico: semaforo de estado del IDS ------------------------
# En la Etapa 6 real esto puede indicar el veredicto del clasificador:
#   verde   -> trafico benigno
#   rojo    -> ataque detectado
#   azul    -> esperando datos
def semaforo(estado):
    """estado in {'benigno', 'ataque', 'esperando'}"""
    mapa = {"benigno": COLORES["verde"], "ataque": COLORES["rojo"],
            "esperando": COLORES["azul"]}
    if rgbleds[5] is not None:
        rgbleds[5].on(mapa.get(estado, 0))
    return estado


for e in ("esperando", "benigno", "ataque", "benigno"):
    print(f"  semaforo -> {semaforo(e)}")
    time.sleep(0.8)

if rgbleds[5] is not None:
    rgbleds[5].off()

## 15 · Bajo nivel: escribir registros AXI con `MMIO`

Todo lo anterior pasó por drivers cómodos. Ahora sacamos la capa de abajo: **`leds[0].on()` termina
siendo una escritura a una dirección de memoria física**.

El IP `AXI GPIO` de Xilinx tiene este mapa de registros:

| Offset | Registro | Función |
|---|---|---|
| `0x00` | `GPIO_DATA` | dato del canal 1 (leer/escribir pines) |
| `0x04` | `GPIO_TRI` | dirección del canal 1: **bit en 0 = salida**, bit en 1 = entrada |
| `0x08` | `GPIO2_DATA` | dato del canal 2 |
| `0x0C` | `GPIO2_TRI` | dirección del canal 2 |

`MMIO(direccion, rango)` mapea esa ventana física en el espacio de usuario del proceso Python (vía
`/dev/mem`), y `write(offset, valor)` es literalmente una transacción AXI hacia la PL.

**Este es el mecanismo exacto** con el que después vas a escribir los registros de control de tu IP de
`hls4ml` (`ap_start`, `ap_done`, direcciones de los buffers). Si entendés esta celda, entendiste el
80 % de cómo se maneja un acelerador desde PYNQ.

In [ ]:
# =====================================================================
# 15. ACCESO DIRECTO A REGISTROS CON MMIO
# =====================================================================
GPIO_DATA, GPIO_TRI = 0x00, 0x04         # mapa de registros del AXI GPIO


def buscar_ip(*claves, excluir=()):
    """Busca en ip_dict: primero coincidencia exacta, despues por prefijo, despues por contenido.

    'excluir' evita falsos positivos: buscando 'led' hay que descartar 'rgbleds_gpio',
    que es otro AXI GPIO distinto (el de los LEDs RGB).
    """
    nombres = [k for k in base.ip_dict
               if not any(x in k.lower() for x in excluir)]
    for c in claves:
        for k in nombres:                       # exacta
            if k.lower() == c:
                return k
    for c in claves:
        for k in nombres:                       # por prefijo
            if k.lower().startswith(c):
                return k
    for c in claves:
        for k in nombres:                       # por contenido
            if c in k.lower():
                return k
    return None


# En el overlay base de la PYNQ-Z2 el IP se llama 'leds_gpio'; la busqueda cubre
# el caso de que tu overlay lo haya nombrado de otra forma.
nombre_ip = buscar_ip("leds_gpio", "led", excluir=("rgb",))
if nombre_ip is None:
    raise RuntimeError("No encontre el AXI GPIO de los LEDs en ip_dict: "
                       f"{list(base.ip_dict)}")
info = base.ip_dict[nombre_ip]
print(f"IP de los LEDs      : {nombre_ip}")
print(f"  direccion fisica  : 0x{info['phys_addr']:08X}")
print(f"  rango             : 0x{info['addr_range']:X}")
print(f"  tipo              : {info.get('type')}")

if EN_PLACA:
    gpio = MMIO(info["phys_addr"], info["addr_range"])

    gpio.write(GPIO_TRI, 0x0)            # los 4 bits como SALIDA
    print(f"\n  GPIO_TRI  <- 0x0  (0 = salida)")

    for patron in (0b0001, 0b0011, 0b0111, 0b1111, 0b1010, 0b0000):
        gpio.write(GPIO_DATA, patron)    # <-- transaccion AXI cruda
        leido = gpio.read(GPIO_DATA)
        clear_output(wait=True)
        print("ESCRITURA DIRECTA POR MMIO\n")
        print(f"  gpio.write(0x{GPIO_DATA:02X}, 0b{patron:04b})")
        print(f"  gpio.read (0x{GPIO_DATA:02X}) -> 0b{leido & 0xF:04b}\n")
        panel(valor_leds=patron, mostrar_entradas=False)
        time.sleep(0.6)

    gpio.write(GPIO_DATA, 0)
    ESTADO_LEDS = 0
    print("\nLEDs apagados por MMIO. Nota: se escribio el MISMO registro que usa leds.write().")
else:
    print("\nSIMULACION: no hay /dev/mem. En la placa, esta celda haria:")
    print("    gpio = MMIO(0x41200000, 0x10000)")
    print("    gpio.write(0x04, 0x0)        # GPIO_TRI  -> salida")
    print("    gpio.write(0x00, 0b1010)     # GPIO_DATA -> LD3 y LD1 encendidos")

## 16 · Buffers contiguos y DMA — el puente hacia el acelerador IDS

Los LEDs se manejan con registros de 32 bits. El clasificador del IDS, en cambio, tiene que recibir un
vector de *features* por cada flujo de red y devolver una probabilidad, a velocidad de línea. Escribir
eso registro por registro sería absurdo, y por eso el diseño de la Etapa 5 usa **AXI-Stream + DMA**.

Tres detalles que hay que entender antes de escribir la primera línea:

1. **Memoria contigua.** El DMA ve direcciones *físicas*, no virtuales. Un `numpy.array` común está
   disperso en páginas: el DMA no lo puede leer. `pynq.allocate()` reserva memoria físicamente contigua
   y devuelve un array que *además* es un `numpy.ndarray` normal.
2. **Coherencia de caché.** El ARM escribe en su caché; el DMA lee la DRAM. Por eso hay que hacer
   `buf.flush()` antes de enviar (bajar la caché a DRAM) y `buf.invalidate()` después de recibir
   (descartar la caché para leer lo que escribió el hardware). Saltarse esto da resultados basura
   *intermitentes*, el peor tipo de bug.
3. **Tipos fijos.** El IP de `hls4ml` espera `ap_fixed<16,6>` (o lo que hayas configurado). El buffer
   se declara con el `dtype` que corresponda al empaquetado que definiste en el envoltorio AXI.

La celda siguiente reserva un buffer real y muestra su dirección física. El bloque de DMA queda como
plantilla comentada, lista para cuando cargues tu overlay.

In [ ]:
# =====================================================================
# 16. BUFFERS CONTIGUOS: PREPARACION PARA EL ACELERADOR
# =====================================================================
import numpy as np

N_FEATURES = 30          # ajustar al numero de features del modelo del IDS

if EN_PLACA:
    buf_in = allocate(shape=(N_FEATURES,), dtype=np.float32)
    buf_out = allocate(shape=(1,), dtype=np.float32)

    buf_in[:] = np.linspace(0, 1, N_FEATURES, dtype=np.float32)

    print("Buffer de entrada")
    print(f"  shape             : {buf_in.shape}   dtype: {buf_in.dtype}")
    print(f"  direccion virtual : {hex(buf_in.ctypes.data)}   (la que ve Python)")
    print(f"  direccion FISICA  : {hex(buf_in.physical_address)}   (la que ve el DMA)")
    print(f"  bytes             : {buf_in.nbytes}")
    print(f"  es un ndarray     : {isinstance(buf_in, np.ndarray)}")
    print(f"  primeros valores  : {np.asarray(buf_in[:5])}")

    buf_in.flush()        # caché del ARM -> DRAM, para que el DMA lea datos frescos
    print("\n  buf_in.flush() ejecutado: los datos ya estan en DDR.")

    buf_in.freebuffer()
    buf_out.freebuffer()
    print("  Buffers liberados (freebuffer).")
else:
    print("SIMULACION: pynq.allocate() no existe fuera de la placa.")
    print(f"En la placa reservaria {N_FEATURES * 4} bytes contiguos y mostraria su direccion fisica.")

# ---------------------------------------------------------------------
# PLANTILLA para la Etapa 6 (descomentar cuando exista el overlay del IDS)
# ---------------------------------------------------------------------
# from pynq import Overlay, allocate
#
# ol  = Overlay("/home/xilinx/jupyter_notebooks/ids_axi.bit")   # + ids_axi.hwh al lado
# dma = ol.axi_dma_0
# ip  = ol.ids_axi_0
#
# buf_in  = allocate(shape=(N_FEATURES,), dtype=np.float32)
# buf_out = allocate(shape=(1,),          dtype=np.float32)
#
# buf_in[:] = features_normalizadas          # el mismo escalado que en el entrenamiento
# buf_in.flush()
#
# ip.write(0x00, 0x01)                       # ap_start
# dma.sendchannel.transfer(buf_in)
# dma.recvchannel.transfer(buf_out)
# dma.sendchannel.wait()
# dma.recvchannel.wait()
#
# buf_out.invalidate()                       # DRAM -> caché del ARM
# probabilidad = float(buf_out[0])
# veredicto = "ATAQUE" if probabilidad >= UMBRAL else "BENIGNO"
# semaforo("ataque" if probabilidad >= UMBRAL else "benigno")   # <- LED RGB de la seccion 14

## 17 · Limpieza final

Buena práctica antes de cerrar el notebook o cargar otro overlay: dejar los periféricos apagados. Los
LEDs mantienen su último valor mientras el bitstream siga cargado, así que si no los apagás se quedan
encendidos indefinidamente.

In [ ]:
# =====================================================================
# 17. LIMPIEZA
# =====================================================================
escribir_leds(0)
for i in (4, 5):
    if rgbleds[i] is not None:
        rgbleds[i].off()

print("Todos los LEDs apagados.\n")
panel("Estado final:")
print("\nEl bitstream sigue cargado en la PL. Para reemplazarlo basta con cargar otro Overlay:")
print('    ol = Overlay("/home/xilinx/jupyter_notebooks/mi_diseño.bit")')

## 18 · Resumen, problemas frecuentes y próximos pasos

### Chuleta de la API que usamos

| Qué querés | Código |
|---|---|
| Cargar el diseño de fábrica | `base = BaseOverlay("base.bit")` |
| Cargar un diseño propio | `ol = Overlay("ruta/diseño.bit")` *(el `.hwh` al lado, mismo nombre)* |
| Ver qué hay en la PL | `base.ip_dict` |
| Encender un LED | `base.leds[0].on()` |
| Escribir los 4 LEDs | `base.leds.write(0b1010, 0xF)` |
| Leer switches / botones | `base.switches.read()` · `base.buttons.read()` |
| Esperar una pulsación | `base.buttons[0].wait_for_value(1)` |
| LED RGB | `base.rgbleds[4].on(0b010)` · `.off()` |
| Registro crudo | `MMIO(dir, rango).write(offset, valor)` |
| Buffer para DMA | `allocate(shape=(N,), dtype=np.float32)` |
| Coherencia de caché | `buf.flush()` antes de enviar · `buf.invalidate()` después de recibir |
| Liberar buffer | `buf.freebuffer()` |

### Problemas frecuentes

| Síntoma | Causa probable | Solución |
|---|---|---|
| No abre `192.168.2.99:9090` | IP del PC mal configurada, o jumper `JP4` no está en `SD` | IP estática `192.168.2.1/24`; revisar jumper y que parpadeen los LEDs al arrancar |
| `ImportError: No module named pynq` | El notebook está corriendo en el PC, no en la placa | Subirlo a la placa (§2). Este notebook igual funciona, en modo simulación |
| `Overlay` tarda mucho o falla | Falta el `.hwh`, o no coincide el nombre con el `.bit` | Copiar ambos archivos, mismo nombre base, misma carpeta |
| Los LEDs no responden | Se cargó otro overlay encima que no cablea los LEDs | Volver a cargar `BaseOverlay("base.bit")` |
| `PermissionError` con `MMIO` | El kernel no corre como root | Usar el usuario `xilinx` estándar de la imagen PYNQ |
| El DMA devuelve basura *a veces* | Falta `flush()` / `invalidate()` | Ver §16, punto 2 |
| El kernel queda colgado en un bucle | Bucle interactivo sin timeout | `Kernel → Interrupt`; los bucles de acá ya tienen timeout |

### Próximos pasos

1. **Compuertas *de verdad* en la FPGA.** Todo lo de §10–§12 se evaluó en el ARM. El ejercicio natural
   siguiente es hacer un overlay mínimo en Vivado donde `LD3` salga de una compuerta cableada entre
   `SW0` y `SW1`: ahí la lógica ocurre en hardware, sin procesador de por medio, y se puede comparar la
   latencia de las dos versiones. Es el "hola mundo" de HDL y encaja perfecto como Etapa 6.b.
2. **Llevar el IP de `hls4ml` a la placa.** Empaquetar el proyecto de la Etapa 5 en Vivado con un
   `axi_dma`, generar `ids_axi.bit` + `ids_axi.hwh`, copiarlos y usar la plantilla de §16.
3. **Validar numéricamente.** Correr el mismo lote de `test_real_traffic.feather` en el hardware y en
   Keras, y comparar las probabilidades: la diferencia es el error de cuantización `ap_fixed`, y debería
   coincidir con lo que ya midió la C-simulation de la Etapa 5.
4. **Medir latencia y throughput reales.** Cronometrar la ronda completa (`flush` + DMA ida + cómputo +
   DMA vuelta + `invalidate`) y contrastarla con el estimado de Vivado HLS y con el baseline de
   `E3_06_simulador_tiempo_real.ipynb`. Ese número es el resultado central del proyecto.
5. **Cerrar el lazo con los LEDs.** El `semaforo()` de §14 ya está listo: verde = benigno,
   rojo = ataque. Es la demo más vendible del trabajo.